# Paper Trading with `time.cell`

An end-to-end trading agent built on top of the `time.cell` and the
`TemporalReasoner` from the Quilt-TimesFM adoption.

## What this notebook shows

1. **Stream a price series** (synthetic GBM by default).
2. **Build a `time.cell`** and bind the rolling history.
3. **Forecast the next N steps** with a trend-aware synthetic model.
4. **Decide a trade** (buy / sell / hold / half_size / gather_data).
5. **Execute on a paper portfolio** with position caps.
6. **Record the actual outcome** when the horizon elapses.
7. **Visualize** the P&L curve and the trade log.

Run all the cells in order. The whole notebook takes ~10 seconds.

In [ ]:
import os
os.environ['QUILT_TIMESFM_SYNTHETIC'] = '1'  # skip the 800MB TimesFM download
import sys
sys.path.insert(0, '..')  # so `from paper_trading import ...` works from the notebooks/ dir
import warnings
warnings.simplefilter('ignore', RuntimeWarning)

import numpy as np
import matplotlib.pyplot as plt

from quilt_cell import TimeCell
from temporal import TemporalReasoner
from paper_trading import (
    PaperTrader, TradingDecisionSupport, TradingAction,
    synthetic_price_stream, EXAMPLE_SHOCKS,
)

## 1. The price stream

We use a Geometric Brownian Motion as the synthetic price source. The
parameters are annualized: `drift=0.10` means ~10% per year expected
return; `vol=0.20` means ~20% per year volatility. The `shocks` list
applies exogenous jumps to simulate news events.

In [ ]:
DRIFT = 0.10
VOL = 0.15
STEPS = 500
SEED = 42
SHOCKS = EXAMPLE_SHOCKS['earnings_beat']  # +10% on day 50, -2% on day 51

stream = synthetic_price_stream(
    n_steps=STEPS, seed=SEED, drift=DRIFT, vol=VOL, shocks=SHOCKS,
)
prices = list(stream)
print(f'first 3 prices: {prices[:3]}')
print(f'last 3 prices:  {prices[-3:]}')
print(f'min / max:      {min(p for _, p in prices):.2f} / {max(p for _, p in prices):.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot([t for t, _ in prices], [p for _, p in prices], color='steelblue', linewidth=1.0)
ax.set_title('Synthetic price stream (Geometric Brownian Motion)')
ax.set_xlabel('step')
ax.set_ylabel('price ($)')
ax.grid(True, alpha=0.3)
plt.show()

## 2. Build the cell + reasoner + strategy

The `TimeCell` is the forecasting substrate. The `TemporalReasoner`
wraps it with the 10 capabilities: scenarios, counterfactuals,
explainability, lifecycle, memory, decisions, URIs, metrics, CRDT.

The `TradingDecisionSupport` is a domain-specific recommender that
translates forecasts into buy / sell / hold / half_size / gather_data.

In [ ]:
cell = TimeCell()
reasoner = TemporalReasoner(cell=cell)
strategy = TradingDecisionSupport(
    memory=reasoner.memory,
    threshold_return=0.005,         # 0.5% expected return to BUY
    threshold_uncertainty=0.4,      # 40% relative CI to HALF_SIZE
)
trader = PaperTrader(
    cell=cell, reasoner=reasoner, strategy=strategy,
    asset='ASSET', history_len=64, horizon=5, min_history=32,
    max_position_pct=0.10,
)

## 3. Run the trader

The `run()` method streams the prices through the cell, generates a
forecast at each tick, makes a decision, executes, and records the
outcome when the horizon elapses.

In [ ]:
stream = synthetic_price_stream(
    n_steps=STEPS, seed=SEED, drift=DRIFT, vol=VOL, shocks=SHOCKS,
)
result = trader.run(stream, verbose=False)
print(f"n_trades    = {result['n_trades']}")
print(f"actions     = {result['n_actions']}")
print(f"final value = ${result['final_value']:,.2f}")
print(f"P&L         = ${result['total_pnl']:+,.2f} ({result['pnl_pct']:+.2%})")
print()
print('First 5 trades:')
for t in result['trade_log'][:5]:
    print(f"  step {t['step']:4d}: {t['action']:11s} @ ${t['current_price']:7.2f}  "
          f"forecast mean = ${t['forecast_mean']:7.2f}  rationale = {t['rationale'][:50]}")

## 4. The P&L curve

Each trade is annotated on the price chart. Buy = green up arrow,
sell = red down arrow, hold = neutral dot.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

# Price + trades
ax1.plot([t for t, _ in prices], [p for _, p in prices], color='steelblue', linewidth=1.0, label='price')
action_styles = {
    'buy':         ('^', 'green', 100),
    'sell':        ('v', 'red', 100),
    'hold':        ('o', 'gray', 30),
    'half_size':   ('s', 'orange', 60),
    'gather_data': ('.', 'lightgray', 10),
}
for t in result['trade_log']:
    if t['action'] not in action_styles:
        continue
    marker, color, size = action_styles[t['action']]
    ax1.scatter(t['step'], t['current_price'], marker=marker, color=color,
                s=size, alpha=0.7, zorder=3)
ax1.set_title(f"Paper trader on {DRIFT:.0%} drift / {VOL:.0%} vol synthetic price (shock: earnings_beat)")
ax1.set_ylabel('price ($)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Cumulative P&L
running_pnl = []
cumulative = 0.0
for t in result['trade_log']:
    if t.get('realized_pnl') is not None:
        cumulative += t['realized_pnl']
    running_pnl.append(cumulative)
ax2.plot([t['step'] for t in result['trade_log']], running_pnl, color='darkgreen', linewidth=1.5)
ax2.axhline(0, color='black', linewidth=0.5, alpha=0.5)
ax2.set_xlabel('step')
ax2.set_ylabel('cumulative P&L ($)')
ax2.set_title(f"Cumulative realized P&L = ${cumulative:+,.2f}")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Calibration over time

The reasoner's `learn_from_history()` reports the model's average
error and calibration. Watch how the calibration improves as more
outcomes are recorded.

In [ ]:
learn = reasoner.memory.learn_from_history('ASSET')
print(f"forecasts made:     {learn['n_forecasts']}")
print(f"outcomes recorded:  {learn['n_recorded_outcomes']}")
print(f"mean error:         {learn.get('mean_error', 'n/a')}")
print(f"mean calibration:   {learn.get('mean_calibration', 'n/a')}")
print(f"error trend:        {learn.get('error_trend', 'n/a')}")
print(f"calibration trend:  {learn.get('calibration_trend', 'n/a')}")

## 6. Counterfactual: what if I had bought at step X?

The reasoner can run a counterfactual forecast: "what would the
model have predicted if the context trend were +20% higher?" Useful
for explaining why a trade was made and for stress-testing the
strategy.

In [ ]:
from temporal import CounterfactualReasoner

# Rebuild the cell with a recent context
ctx = np.array(prices[200:264, 1])  # 64 prices ending at step 264
cell.bind_context(ctx.reshape(-1, 1))
cell.set_horizon(5)
cell.forecast_trend()
baseline = cell.read_point(0)

cf = CounterfactualReasoner(cell)
results = {
    'baseline':         baseline.tolist(),
    'trend +20%':       cf.counterfactual('context_trend', 0.20)['point'].tolist(),
    'trend -20%':       cf.counterfactual('context_trend', -0.20)['point'].tolist(),
    'volatility +50%':  cf.counterfactual('context_volatility', 0.50)['point'].tolist(),
    'volatility -50%':  cf.counterfactual('context_volatility', -0.50)['point'].tolist(),
}

fig, ax = plt.subplots(figsize=(12, 4))
for label, forecast in results.items():
    style = '-' if label == 'baseline' else '--'
    width = 2.0 if label == 'baseline' else 1.0
    ax.plot(range(1, len(forecast) + 1), forecast, style, linewidth=width, label=label)
ax.axhline(ctx[-1], color='black', linestyle=':', alpha=0.5, label='current price')
ax.set_title('Counterfactual forecasts: how does the prediction change?')
ax.set_xlabel('steps ahead')
ax.set_ylabel('predicted price ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 7. The quf:// URI scheme

Every forecast has a unique addressable URI. This makes the trade log
CRDT-mergeable across multiple agents.

In [ ]:
print('First 3 quf:// URIs from the trade log:')
for t in result['trade_log'][:3]:
    print(f"  step {t['step']:4d}: {t['forecast_uri']}")
print()
print('All URIs are unique (CRDT-mergeable):')
uris = [t['forecast_uri'] for t in result['trade_log']]
print(f'  {len(uris)} trades, {len(set(uris))} unique URIs')

## 8. Try it with a real TimesFM model

If you have an 800MB TimesFM 3.0 checkpoint available, unset the env
var and rerun. The cell will call the real model instead of the
synthetic trend forecast.

```bash
# unset the synthetic flag
unset QUILT_TIMESFM_SYNTHETIC
python -m paper_trading --steps 500 --shock earnings_beat
```